# 06 — Telephony — Twilio

Webhook parsing, HMAC-SHA1, AMD, DTMF, recording, transfer, ring timeout, status callback, TwiML emission.


## Prerequisites

| Tier | Cells | Required env |
|------|-------|--------------|
| T1+T2 (§1) | always | _none_ |
| T3 (§2) | per-cell | provider keys auto-detected |
| T4 (§3) | gated | `ENABLE_LIVE_CALLS=1` + carrier creds |


In [ ]:
import { load } from "./_setup.ts";
const env = load();
console.log(`getpatter version target: ${env.patterVersion}`);


## §1: Quickstart

Runs end-to-end with zero API keys.


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


In [ ]:
import { cell } from "./_setup.ts";
import * as getpatter from "getpatter";
await cell('version_check', { tier: 1, env }, () => {
  const v = (getpatter as any).version ?? (getpatter as any).VERSION ?? 'unknown';
  console.log(`getpatter ${v}`);
});


### Local mode
Construct a Patter instance with a Twilio carrier.


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('local_mode', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({
      accountSid: 'ACtest00000000000000000000000000',
      authToken: 'test',
    }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Patter local mode constructed OK');
});


### Cloud mode (coming soon)
When `apiKey` is supported, Patter cloud handles telephony. For now, the SDK throws — this cell verifies the guard.


In [ ]:
import { Patter } from "getpatter";
await cell('cloud_mode', { tier: 1, env }, () => {
  try {
    new Patter({ apiKey: 'pt_test_xxx' } as any);
    throw new Error('expected error — cloud mode guard missing');
  } catch (e: any) {
    if (e.message?.includes('Cloud') || e.message?.includes('cloud') || e.message?.includes('apiKey')) {
      console.log(`cloud mode guard OK: ${e.message.slice(0, 80)}`);
    } else { throw e; }
  }
});


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


In [ ]:
import { Patter, Twilio, OpenAIRealtime, ElevenLabsConvAI } from "getpatter";
await cell('agent_engines', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  const rt = p.agent({ systemPrompt: 'hi', engine: new OpenAIRealtime({ apiKey: 'sk-test' }) });
  const cv = p.agent({ systemPrompt: 'hi', engine: new ElevenLabsConvAI({ apiKey: 'el-test', agentId: 'a1' }) });
  const pl = p.agent({ systemPrompt: 'hi' });
  if (rt.provider !== 'openai_realtime') throw new Error(`expected openai_realtime, got ${rt.provider}`);
  if (cv.provider !== 'elevenlabs_convai') throw new Error(`expected elevenlabs_convai, got ${cv.provider}`);
  console.log(`rt=${rt.provider}  cv=${cv.provider}  pl=${pl.provider}`);
});


These cells run with **zero API keys** in <30 seconds. They exercise the public Patter API offline.


### Local mode
Construct a Patter instance with a Twilio carrier.


### Cloud mode
Same SDK, just an `apiKey` — Patter cloud handles telephony.


### Three engine types
An agent picks one of *OpenAI Realtime*, *ElevenLabs ConvAI*, or *Pipeline*.


## §2: Feature Tour

One cell per feature/provider. Missing keys auto-skip.


## §2 — Feature Tour

Exercises Twilio webhook signature verification and carrier construction.


### Twilio signature verification — valid request


In [ ]:
import crypto from 'crypto';
await cell('twilio_sig_valid', { tier: 1, env }, () => {
  const authToken = 'test_auth_token_32chars_padding___';
  const url       = 'https://example.com/webhook/voice';
  const params    = { CallSid: 'CA0000000000000000000000000000a001', From: '+15555550100' };
  const sorted    = Object.keys(params).sort().map(k => k + (params as any)[k]).join('');
  const sig       = crypto.createHmac('sha1', authToken).update(url + sorted).digest('base64');
  console.log(`Twilio signature: ${sig}`);
  console.log('Signature computed OK (validate against Twilio SDK in production)');
});


### Twilio signature verification — tampered request


In [ ]:
import crypto from 'crypto';
await cell('twilio_sig_invalid', { tier: 1, env }, () => {
  const authToken = 'test_auth_token_32chars_padding___';
  const url       = 'https://example.com/webhook/voice';
  const params    = { CallSid: 'CA0000000000000000000000000000a001', From: '+15555550100' };
  const sorted    = Object.keys(params).sort().map(k => k + (params as any)[k]).join('');
  const goodSig   = crypto.createHmac('sha1', authToken).update(url + sorted).digest('base64');
  const badSig    = 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA=';
  console.log(`Good sig: ${goodSig.slice(0, 20)}...`);
  console.log(`Bad  sig: ${badSig.slice(0, 20)}...`);
  console.log(`Tampered signature differs: ${goodSig !== badSig}  (would be rejected)`);
  if (goodSig === badSig) throw new Error('signatures should differ');
});


### E.164 phone number patterns


In [ ]:
await cell('e164_patterns', { tier: 1, env }, () => {
  const E164_RE = /^\+[1-9]\d{6,14}$/;
  const cases: [string, boolean][] = [
    ['+15555550100', true], ['+442071838750', true],
    ['15555550100', false], ['+1', false],
  ];
  for (const [num, expected] of cases) {
    const result = E164_RE.test(num);
    const ok = result === expected ? '✓' : '✗';
    console.log(`  ${ok} ${num.padEnd(16)} → ${result}`);
  }
});


### Twilio carrier construction


In [ ]:
import { Patter, Twilio } from "getpatter";
await cell('twilio_carrier', { tier: 1, env }, () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: 'ACtest00000000000000000000000000', authToken: 'test' }),
    phoneNumber: '+15555550100',
    webhookUrl: 'https://example.com/webhook',
  });
  console.log('Twilio carrier constructed OK');
});


## §3: Live Appendix

Real PSTN calls. Off by default — set `ENABLE_LIVE_CALLS=1`.


## §3 — Live Appendix

Tests Twilio call flow including AMD detection. Requires `ENABLE_LIVE_CALLS=1`.


### Pre-flight checklist


In [ ]:
await cell('live_preflight', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER'], env }, () => {
  console.log(`  carrier:  Twilio ${env.twilioNumber}  →  ${env.targetNumber}`);
  console.log(`  webhook:  ${env.publicWebhookUrl || '(ngrok auto-launch)'}`);
  console.log('  features: AMD + voicemail fallback');
});


### Live Twilio call with AMD *(T4)*


In [ ]:
import { Patter, Twilio, OpenAIRealtime } from "getpatter";
await cell('live_twilio_amd', { tier: 4, required: ['TWILIO_ACCOUNT_SID', 'TWILIO_AUTH_TOKEN', 'TWILIO_PHONE_NUMBER', 'TARGET_PHONE_NUMBER', 'OPENAI_API_KEY'], env }, async () => {
  const p = new Patter({
    carrier: new Twilio({ accountSid: env.twilioSid, authToken: env.twilioToken }),
    phoneNumber: env.twilioNumber,
    webhookUrl: env.publicWebhookUrl,
  });
  const agent = p.agent({
    systemPrompt: 'Twilio demo agent. Greet and hang up.',
    engine: new OpenAIRealtime({ apiKey: env.openaiKey }),
  });
  try {
    await p.call(env.targetNumber, {
      agent, machineDetection: true,
      voicemailMessage: 'Patter demo voicemail.',
      firstMessage: 'Hello from Patter Twilio.',
      ringTimeout: env.maxCallSeconds,
    });
    console.log('✓ Twilio AMD call completed');
  } finally {
    await hangupLeftoverCalls(env);
  }
});
